In [ ]:
import torch
import torch.nn as nn

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self,d_in,d_out):
        self.__init__()
        self.query_w=nn.Linear(d_in,d_out,bias=False)
        self.key_w=nn.Linear(d_in,d_out,bias=False)
        self.value_w=nn.Linear(d_in,d_out,bias=False)
    def forward(self,X):
        queries=self.query_w(X)
        keys=self.key_w(X)
        values=self.value_w(X)

        attn_scores=queries@keys.T
        attn_scores=torch.softmax(attn_scores/(keys.shape[-1]**0.5),dim=-1)
        context_values=attn_scores@values

        return context_values



In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self,d_in,d_out,num_heads,dropout):
        self.__init__()
        self.query_w=self.linear(d_in,d_out,bias=False)
        self.key_w=self.linear(d_in,d_out,bias=False)
        self.value_w=self.linear(d_in,d_out,bias=False)
        self.head_dim=d_out//num_heads
        self.dropout=nn.Dropout(dropout)
    
    def forward(self,X):
        batch,num_tokens,d_in=X.shape

        query=self.query_w(X)
        keys=self.key_w(X)
        value=self.value_w(X)


        keys=keys.view(batch,num_tokens,self.num_heads,self.head_dim)
        query=query.view(batch,num_tokens,self.num_heads,self.head_dim)
        value=value.view(batch,num_tokens,self.num_heads,self.head_dim)
        
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores=torch.softmax(attn_scores/(keys.shape[-1]**0.5),dim=-1)
        context_vec = (attn_scores @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(batch, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec
